In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.multicomp import MultiComparison
from scipy import stats

In [ ]:
# config
# as environment: ../../candidate-profling/cand-prof-env.yml
PROTEIN_NAME = "vcam1"
PROTEIN_NAME_UPPER = PROTEIN_NAME.upper()
DATA = f"data/crispr-cas9-{PROTEIN_NAME}-hippo-intensity.xlsx"

In [ ]:
df = pd.read_excel(DATA)
df.head(10)

### Data transformation

In [ ]:
# Subtract background per animal
bg = df[df['condition'] == 'background'].set_index('Brain')['raw_mean_intensity']

wide = df[df['condition'] != 'background'].pivot(index='Brain', columns='condition', values='raw_mean_intensity')
wide['background'] = bg
wide['LacZ_corrected'] = wide['LacZ-gRNA'] - wide['background']
wide[f"{PROTEIN_NAME_UPPER}_corrected"] = wide[f"{PROTEIN_NAME_UPPER}-gRNA"] - wide['background']

# relative expression, normalized to mean of LacZ controls
wide['LacZ_relative'] = wide['LacZ_corrected'] / wide['LacZ_corrected'].mean()
wide[f"{PROTEIN_NAME_UPPER}_relative"] = wide[f"{PROTEIN_NAME_UPPER}_corrected"] / wide['LacZ_corrected'].mean()

wide

### Plotting the data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- data prep (from your `wide` dataframe) ---
lacz = wide['LacZ_relative'].values
protein_of_interest = wide[f"{PROTEIN_NAME_UPPER}_relative"].values

groups = [lacz, protein_of_interest]
labels = ['LacZ-gRNA', f"{PROTEIN_NAME_UPPER}-gRNA"]

if PROTEIN_NAME == "vcam1":
    dot_colors = ['#4d4d4d', '#3a7bd5']  
    colors = ['#d9d9d9', '#BFEBFF']
else:
    dot_colors = ['#4d4d4d', '#E28D21']
    colors = ['#d9d9d9', '#F8E2C7']     # dot color per group
x_pos = [0, 1]

fig, ax = plt.subplots(figsize=(2.2, 4.5))

# bars = group means
ax.bar(x_pos[0], np.mean(lacz), width=0.6, color=colors[0], zorder=1)
ax.bar(x_pos[1], np.mean(protein_of_interest), width=0.6, color=colors[1], zorder=1)

# connecting lines (straight, no jitter)
for y0, y1 in zip(lacz, protein_of_interest):
    ax.plot([x_pos[0], x_pos[1]], [y0, y1], color='gray', linewidth=2, alpha=0.6, zorder=2)

# dots
ax.scatter(np.full(len(lacz), x_pos[0]), lacz, color=dot_colors[0], s=100, zorder=3,
           edgecolor='black', linewidth=1)
ax.scatter(np.full(len(protein_of_interest), x_pos[1]), protein_of_interest, color=dot_colors[1], s=100, zorder=3,
           edgecolor='black', linewidth=1)

# --- formatting ---
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, rotation=0)
ax.set_ylabel('relative VCAM1 protein levels\n(norm. to LacZ-gRNA)')
ax.set_ylim(0, 2)
ax.set_yticks(np.arange(0, 2.01, 1))
ax.set_xlim(-0.5, 1.5)
ax.spines[['top', 'right']].set_visible(False)

# thicker axis spines and ticks
ax.spines['left'].set_linewidth(2)
ax.spines['bottom'].set_linewidth(2)
ax.tick_params(axis='x', length=0, width=2)
ax.tick_params(axis='y', length=7, width=2)

plt.tight_layout()
plt.show()

fig.savefig(
    f"{PROTEIN_NAME}_KO_analysis.svg",
    format="svg",
    bbox_inches="tight"
)

### Statistical test: paired t-test

In [ ]:
t_stat, p_val = stats.ttest_rel(wide['LacZ_corrected'], wide[f"{PROTEIN_NAME_UPPER}_corrected"])
print(f"paired t-test: t={t_stat:.3f}, p={p_val:.4g}")